##DATA Ingestion

In [1]:
from pathlib import Path

file_path = Path("data/products.txt")

text = file_path.read_text()

print("Data loaded successfully")
print("=" * 50)
print(text)
print("=" * 50)
print("Characters:", len(text))

Data loaded successfully
Product: iPhone 17 Pro
Category: Smartphone
Features: 5G, USB-C, wireless charging, Face ID.
Price: $1199.

Product: AirPods Pro 3
Category: Earbuds
Features: Active Noise Cancellation, wireless charging, Bluetooth.
Price: $249.

Product: USB-C Cable
Category: Accessory
Features: Fast charging and high-speed data transfer.
Price: $29.
Characters: 336


In [ ]:
print(type(text)) #before load data is string 

<class 'str'>


Stage-2 Chunking process -Now we break large text into smaller piecec 

In [5]:
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = Path("data/products.txt").read_text()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=20
)

chunks = splitter.split_text(text)

print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print("\n" + "=" * 50)
    print("CHUNK", i + 1)
    print("=" * 50)
    print(chunk)

Total chunks: 11

CHUNK 1
Product: iPhone 17 Pro
Category: Smartphone

CHUNK 2
Features: 5G, USB-C, wireless charging, Face ID.

CHUNK 3
Price: $1199.

CHUNK 4
Product: AirPods Pro 3
Category: Earbuds

CHUNK 5
Features: Active Noise Cancellation, wireless

CHUNK 6
wireless charging, Bluetooth.

CHUNK 7
Price: $249.

CHUNK 8
Product: USB-C Cable
Category: Accessory

CHUNK 9
Features: Fast charging and high-speed data

CHUNK 10
and high-speed data transfer.

CHUNK 11
Price: $29.


Stage 4 — Create embeddings TEXT to Numbers 

In [7]:
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv()

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

print("Gemini embedding model initialized")

Gemini embedding model initialized


In [9]:
#Apply embedding to chunks
for i, chunk in enumerate(chunks):

    vector = embeddings.embed_query(chunk)

    print("=" * 60)
    print("CHUNK:", i + 1)

    print("\nChunk text:")
    print(chunk)

    print("\nVector type:")
    print(type(vector))

    print("\nVector dimensions:")
    print(len(vector))

    print("\nFirst 5 values:")
    print(vector[:5])

CHUNK: 1

Chunk text:
Product: iPhone 17 Pro
Category: Smartphone

Vector type:
<class 'list'>

Vector dimensions:
3072

First 5 values:
[-0.011693265, 0.015476741, 0.0037819676, -0.07150735, 0.00046411017]
CHUNK: 2

Chunk text:
Features: 5G, USB-C, wireless charging, Face ID.

Vector type:
<class 'list'>

Vector dimensions:
3072

First 5 values:
[0.008924323, -0.016242638, -0.017365852, -0.052791405, 0.0043556695]
CHUNK: 3

Chunk text:
Price: $1199.

Vector type:
<class 'list'>

Vector dimensions:
3072

First 5 values:
[-0.010227832, 0.012679871, 0.011439642, -0.079370625, -0.0011021371]
CHUNK: 4

Chunk text:
Product: AirPods Pro 3
Category: Earbuds

Vector type:
<class 'list'>

Vector dimensions:
3072

First 5 values:
[0.0029492504, 0.0019653242, 0.026777016, -0.06450022, 0.016033053]
CHUNK: 5

Chunk text:
Features: Active Noise Cancellation, wireless

Vector type:
<class 'list'>

Vector dimensions:
3072

First 5 values:
[-0.001850103, -0.01879319, 0.0007382924, -0.059562918, -0.0236

In [10]:
# store in chroma
from langchain_core.documents import Document
from langchain_chroma import Chroma

docs = [
    Document(page_content=chunk)
    for chunk in chunks
]

print("Documents created:", len(docs))

Documents created: 11


In [12]:
#Create Vector Database
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="products"
)

print("Vector database created successfully")

Vector database created successfully


In [ ]:
#verification
print("Chunks:", len(chunks))
print("Documents:", len(docs))

Chunks: 11
Documents: 11


In [14]:
question = "Which product supports 5G?"

results = vectorstore.similarity_search(question, k=3)

for doc in results:
    print("=" * 50)
    print(doc.page_content)

Features: 5G, USB-C, wireless charging, Face ID.
Product: iPhone 17 Pro
Category: Smartphone
and high-speed data transfer.


Question-2

In [15]:
question = "Which product has active noise cancellation?"

results = vectorstore.similarity_search(question, k=3)

for doc in results:
    print("=" * 50)
    print(doc.page_content)

Features: Active Noise Cancellation, wireless
Product: AirPods Pro 3
Category: Earbuds
Product: iPhone 17 Pro
Category: Smartphone
